In [1]:
import numpy as np
import pandas as pd

# 1. Ingestión forzada mapeando textos vacíos como nulos reales
df = pd.read_csv('hantavirus_QA_VERIFICADO.csv', na_values=["NA", "na", "NaN", "nan", " ", ""])

# 2. Renombrado corporativo de columnas
df.rename(columns={
    "cruise.crew..y.n.": "is_crew", 
    "passenger..y.n.": "is_passenger", 
    "Gh_ID": "gh_id",
    "Travel_flight": "travel_flight"
}, inplace=True)

# 3. Tratamiento de Registros Vacíos y Errores en Categóricas
df["status"] = df["status"].fillna("unspecified").astype(str).str.lower().str.strip()
df["sex"] = df["sex"].fillna("unknown").astype(str).str.lower().str.strip()
df["nationality"] = df["nationality"].fillna("unknown").astype(str).str.lower().str.strip()

# 4. Corrección de la Regla de Negocio: Hospitalización Correcta
# Solo 'confirmed' y 'probable' se hospitalizan. Los demás conservan su estado original o van a monitoreo.
is_critical = df["status"].isin(["confirmed", "probable"])
df["treatment"] = np.where(is_critical, "hospitalised", df["treatment"].astype(str).str.lower().str.strip())
df["treatment"] = np.where((df["treatment"] == "hospitalised") & (~is_critical), "monitored", df["treatment"])

# 5. Corrección de la columna Confirmed (Binario Seguro)
df["confirmed"] = np.where(is_critical, "yes", "no")

# 6. Solución Científica para la Edad (Mediana e Imputación de rango)
df["age"] = df["age"].astype(str).str.lower().str.strip()
# Si contiene 's' (como 70s), extraemos el número y le sumamos 5 (punto medio de la década)
has_s = df["age"].str.contains("s", na=False)
df["age"] = pd.to_numeric(df["age"].str.extract(r"(\d+)", expand=False), errors="coerce")
df.loc[has_s, "age"] += 5

# Calcular mediana de la edad para registros vacíos
mediana_edad = df["age"].median()
df["age"].fillna(round(mediana_edad), inplace=True)
df["age"] = df["age"].astype("Int64")

# 7. Imputación Temporal (Incubación de 18 días)
df["ship_boarded"] = pd.to_datetime("2026-04-01")
df["symptom_onset"] = pd.to_datetime(df["symptom_onset"], errors="coerce")
df["symptom_onset"].fillna(df["ship_boarded"] + pd.Timedelta(days=18), inplace=True)

# 8. Desglose atómico de síntomas (One-Hot Encoding)
df["symptoms"] = df["symptoms"].fillna("no symptoms").astype(str).str.lower().str.strip()
df["symptoms"].replace({"diarreah": "diarrhea", "diarrhiea": "diarrhea"}, regex=True, inplace=True)

all_symptoms = df["symptoms"].str.split(",").explode().str.strip()
unique_symptoms = all_symptoms[(all_symptoms != "") & (all_symptoms != "no symptoms") & (~all_symptoms.str.contains(r"\(", na=False))].unique()

for symptom in unique_symptoms:
    col_name = f"has_{symptom.replace(' ', '_').replace('-', '_')}"
    df[col_name] = df["symptoms"].str.contains(symptom, regex=False).astype(int)

# 9. Eliminación final de columnas con ruido de alta cardinalidad
columnas_ruido = ["linked_information", "context", "sources", "accession_id", "symptoms"]
df.drop(columns=columnas_ruido, errors="ignore", inplace=True)

# 10. Formateo de IDs remanentes
df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)

# Guardado definitivo
df.to_csv("hantavirus_PRODUCTION_READY.csv", index=False)

print("🚀 ¡DATASET BLINDADO Y PERSISTIDO EN DISCO!")
print(f"Dimensiones finales: {df.shape[0]} filas x {df.shape[1]} columnas.")
print(f"Columnas limpias listas para BI: {list(df.columns)}")

🚀 ¡DATASET BLINDADO Y PERSISTIDO EN DISCO!
Dimensiones finales: 20 filas x 35 columnas.
Columnas limpias listas para BI: ['gh_id', 'linked_id', 'status', 'symptom_onset', 'outcome', 'outcome_date', 'treatment', 'treatment_date', 'is_crew', 'is_passenger', 'age', 'sex', 'cause_death', 'exposure', 'travel_from', 'travel_to', 'travel_flight', 'ship_boarded', 'left_ship', 'left_location', 'confirmed', 'confirmation_date', 'nationality', 'has_fever', 'has_headache', 'has_diarrhea', 'has_shortness_of_breath', 'has_signs_of_pneumonia', 'has_flu_like_symptoms', 'has_later_signs_of_pneumonia', 'has_mild_symptoms', 'has_runny_nose', 'has_diagnosed_with_symptoms_consistent_with_a_hantavirus_infection', 'has_respiratory_symptoms', 'has_pneumonia']


In [2]:
def apply_cause_death_protocol(file_path: str) -> pd.DataFrame:
    # 1. Ingestión del dataset mapeando nulos iniciales
    df = pd.read_csv(file_path, na_values=["NA", "na", "NaN", "nan", " ", ""], keep_default_na=True)
    
    # 2. Estandarización de columnas clave para el cruce lógico
    df["outcome"] = df["outcome"].astype(str).str.lower().str.strip().replace("nan", np.nan)
    df["cause_death"] = df["cause_death"].astype(str).str.lower().str.strip().replace("nan", np.nan)
    
    # =============================================================================
    # REGLA DE NEGOCIO CLÍNICA: IMPUTACIÓN DE CAUSA DE MUERTE POR HANTAVIRUS
    # =============================================================================
    # Condición A: El paciente falleció (outcome == 'died')
    has_died = df["outcome"] == "died"
    
    # Condición B: La causa de muerte está en blanco (isnull)
    cause_is_blank = df["cause_death"].isnull()
    
    # Aplicamos el protocolo epidemiológico:
    # Si el paciente falleció y la causa estaba en blanco, asignamos el Síndrome Pulmonar.
    # Si ya tenía un registro previo (como 'respiratory distress'), lo preservamos.
    df["cause_death"] = np.where(
        has_died & cause_is_blank, 
        "hantavirus pulmonary syndrome", 
        df["cause_death"]
    )
    
    # Para los pacientes que NO fallecieron, nos aseguramos de limpiar la columna 
    # asignando una etiqueta de control limpia ("not applicable") para evitar NaN molestos en Power BI.
    df["cause_death"] = np.where(
        ~has_died, 
        "not applicable", 
        df["cause_death"]
    )

    # =============================================================================
    # Bloque de Consolidación y Limpieza Radical (Sprints Anteriores)
    # =============================================================================
    df.rename(columns={"cruise.crew..y.n.": "is_crew", "passenger..y.n.": "is_passenger", "Gh_ID": "gh_id"}, inplace=True)
    df["status"] = df["status"].fillna("unspecified").astype(str).str.lower().str.strip()
    
    # Corrección de Hospitalización y Flags
    is_critical = df["status"].isin(["confirmed", "probable"])
    df["treatment"] = np.where(is_critical, "hospitalised", df["treatment"].astype(str).str.lower().str.strip())
    df["confirmed"] = np.where(is_critical, "yes", "no")
    
    # Edad a Entero Puro (Int64)
    df["age"] = df["age"].astype(str).str.lower().str.strip()
    has_s = df["age"].str.contains("s", na=False)
    df["age"] = pd.to_numeric(df["age"].str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5
    df["age"].fillna(round(df["age"].median()), inplace=True)
    df["age"] = df["age"].astype("Int64")
    
    # Desglose de síntomas (One-Hot Encoding)
    df["symptoms"] = df["symptoms"].fillna("no symptoms").astype(str).str.lower().str.strip()
    all_symptoms = df["symptoms"].str.split(",").explode().str.strip()
    unique_symptoms = all_symptoms[(all_symptoms != "") & (all_symptoms != "no symptoms") & (~all_symptoms.str.contains(r"\(", na=False))].unique()
    for symptom in unique_symptoms:
        df[f"has_{symptom.replace(' ', '_').replace('-', '_')}"] = df["symptoms"].str.contains(symptom, regex=False).astype(int)
        
    # Eliminación de ruido y guardado persistente
    columnas_ruido = ["linked_information", "context", "sources", "accession_id", "symptoms"]
    df.drop(columns=columnas_ruido, errors="ignore", inplace=True)
    
    return df

# Ejecución y Auditoría Médica en tiempo real
if __name__ == "__main__":
    PATH_INPUT = "hantavirus_QA_VERIFICADO.csv"
    df_finalizado = apply_cause_death_protocol(PATH_INPUT)
    
    print("📋 ### AUDITORÍA DE PACIENTES FALLECIDOS (POST-REGLA) ###")
    # Filtramos para ver cómo quedaron los registros de fallecidos
    df_fallecidos = df_finalizado[df_finalizado["outcome"] == "died"]
    print(df_fallecidos[["gh_id", "status", "outcome", "cause_death"]].to_string(index=False))
    
    print("\n" + "="*70 + "\n")
    print("📊 ### CONTEO DE CATEGORÍAS EN CAUSE_DEATH ###")
    print(df_finalizado["cause_death"].value_counts())
    
    # Guardar los cambios físicos de forma definitiva
    df_finalizado.to_csv("hantavirus_PRODUCTION_READY.csv", index=False)
    print("\n Archivo 'hantavirus_PRODUCTION_READY.csv' actualizado y guardado.")

📋 ### AUDITORÍA DE PACIENTES FALLECIDOS (POST-REGLA) ###
 gh_id    status outcome                   cause_death
     1  probable    died          respiratory distress
     2 confirmed    died hantavirus pulmonary syndrome
     4 confirmed    died hantavirus pulmonary syndrome


📊 ### CONTEO DE CATEGORÍAS EN CAUSE_DEATH ###
cause_death
not applicable                   17
hantavirus pulmonary syndrome     2
respiratory distress              1
Name: count, dtype: int64

 Archivo 'hantavirus_PRODUCTION_READY.csv' actualizado y guardado.


In [3]:
def transform_symptoms_to_text(file_path: str) -> pd.DataFrame:
    # 1. Cargamos el dataset que ya incluye el procesamiento previo
    # (Asegúrate de apuntar al archivo que generamos en el paso anterior)
    df = pd.read_csv(file_path)
    
    # =============================================================================
    # TRANSFORMAR COLUMNAS 'has_' DE BINARIO (1/0) A TEXTO (yes/no)
    # =============================================================================
    # A. Identificamos de forma dinámica todas las columnas que empiezan con 'has_'
    symptom_columns = [col for col in df.columns if col.startswith("has_")]
    
    print(f"📢 Columnas de síntomas detectadas para transformación: {symptom_columns}")
    
    # B. Iteramos sobre cada columna detectada para hacer la sustitución
    for col in symptom_columns:
        # Reemplazamos 1 por 'yes' y 0 por 'no'
        # Usamos .map() o np.where() asegurando que se guarden como tipo texto (str)
        df[col] = df[col].map({1: "yes", 0: "no"}).astype(str)
    
    return df

# Ejecución y control de calidad inmediato
if __name__ == "__main__":
    # Usamos el último archivo generado en la sesión
    PATH_INPUT = "hantavirus_PRODUCTION_READY.csv" 
    
    # Ejecutamos la transformación
    df_visualizacion = transform_symptoms_to_text(PATH_INPUT)
    
    print("\n" + "="*70 + "\n")
    print("📋 ### AUDITORÍA DE FORMATO CATEGÓRICO (Primeros 5 registros) ###")
    
    # Busquemos un par de columnas de síntomas para mostrar el control de calidad
    symptom_cols = [col for col in df_visualizacion.columns if col.startswith("has_")]
    columns_to_show = ["gh_id", "status"] + symptom_cols[:4] # Mostramos los primeros 4 síntomas
    
    print(df_visualizacion[columns_to_show].head(5).to_string(index=False))
    
    # =============================================================================
    # GUARDADO DEL ARCHIVO FINAL DE PRODUCCIÓN
    # =============================================================================
    # Mantenemos el mismo nombre para consolidar toda la jerarquía de limpieza
    df_visualizacion.to_csv("hantavirus_PRODUCTION_READY.csv", index=False)
    print("\n✅ ¡Sustitución completada con éxito!")
    print("💾 El archivo 'hantavirus_PRODUCTION_READY.csv' ha sido actualizado y está 100% optimizado para tus dashboards.")

📢 Columnas de síntomas detectadas para transformación: ['has_fever', 'has_headache', 'has_diarrhea', 'has_diarreah', 'has_shortness_of_breath', 'has_signs_of_pneumonia', 'has_flu_like_symptoms', 'has_later_signs_of_pneumonia', 'has_mild_symptoms', 'has_runny_nose', 'has_diarrhiea', 'has_diagnosed_with_symptoms_consistent_with_a_hantavirus_infection', 'has_respiratory_symptoms', 'has_pneumonia']


📋 ### AUDITORÍA DE FORMATO CATEGÓRICO (Primeros 5 registros) ###
 gh_id    status has_fever has_headache has_diarrhea has_diarreah
     1  probable       yes          yes          yes           no
     2 confirmed        no           no           no          yes
     3 confirmed       yes           no           no           no
     4 confirmed        no           no           no           no
     5 confirmed        no           no           no           no

✅ ¡Sustitución completada con éxito!
💾 El archivo 'hantavirus_PRODUCTION_READY.csv' ha sido actualizado y está 100% optimizado para tus da